# 3 - Inserting Metadata

The code of the tutorial page "3 - Inserting Metadata".
Run `1_inserting_data` first, in the same folder.

In [1]:
import acquirium as aq

acq = aq.init()

## Built-in keys

In [2]:
acq.register_streams([{
    "source_id": "lab-import",
    "stream_name": "effluent-tds",
    "unit": "mg/L",
    "quantity_kind": "mass concentration",
}])

A text that does not match anything is skipped with a warning:

In [3]:
acq.register_streams([{
    "source_id": "lab-import",
    "stream_name": "effluent-ph",
    "unit": "pH units",
}])

acquirium/Client/client.py:1081: UserWarning: Could not resolve 'unit' value 'pH units' to a QUDT URI; skipping


The symbol of the unit is matched:

In [4]:
acq.register_streams([{
    "source_id": "lab-import",
    "stream_name": "effluent-ph",
    "unit": "pH",
}])

## Your own keys

In [5]:
ph = acq.stream("lab-import", "effluent-ph")

acq.insert_metadata(ph, {"sampled_by": "day shift"})

{'ok': True, 'message': 'update applied', 'changed': True, 'nodes': 1}

Your own keys can also be given at registration, with the `metadata` key:

In [7]:
acq.register_streams([{
    "source_id": "lab-import",
    "stream_name": "effluent-tds",
    "metadata": {"sampled_by": "night shift"},
}])

## Seeing the metadata

In [8]:
q = acq.query().measurement()

q.include("unit", "quantity_kind", "sampled_by").metadata()

data,data.label,data.unit,data.quantity_kind,data.sampled_by
str,str,str,str,str
"""acq:b67d0b22-3ac0-5060-9618-d9…","""effluent-tds""","""unit:MilliGM-PER-L""","""quantitykind:MassConcentration""","""night shift"""
"""acq:56d39005-89a9-50a3-9b51-f2…","""effluent-ph""","""unit:PH""",null,"""day shift"""


## Selecting streams by metadata

By a built-in key:

In [9]:
q.where(quantity_kind="mass concentration").dataframe()

time,data__effluent-tds
"datetime[μs, UTC]",f64
2026-08-08 10:00:00 UTC,412.0
2026-08-08 10:05:00 UTC,415.3
2026-08-08 10:10:00 UTC,418.1


By your own key:

In [10]:
q.where(sampled_by="day shift").dataframe()

time,data__effluent-ph
"datetime[μs, UTC]",f64
2026-08-08 10:10:00 UTC,7.2


Several filters:

In [11]:
q.where(unit="mg/L", sampled_by="night shift").metadata()

data,data.label
str,str
"""acq:b67d0b22-3ac0-5060-9618-d9…","""effluent-tds"""


The values a key has:

In [12]:
q.options("sampled_by")

sampled_by,count
str,i64
"""day shift""",1
"""night shift""",1


## Converting units

In [13]:
q.where(unit="mg/L").data().convert_to("g/L").dataframe()

time,data__effluent-tds
"datetime[μs, UTC]",f64
2026-08-08 10:00:00 UTC,0.412
2026-08-08 10:05:00 UTC,0.4153
2026-08-08 10:10:00 UTC,0.4181


Stop Acquirium.

In [14]:
aq.shutdown()